# GeoIPS Output Formatters Tutorial
Previously you learned how to create a new plugin, now we will practice editing the output to fit your needs!

## Recap of the beginner tutorial
Earlier, you produced a plugin which uses workflows to generate imagery of severe storms.

Run the following command and check whether that package is still there

In [ ]:
%%bash

# Verification
geoips list packages

Confirm that `cool_plugins` is present in the list above.

If not, you should revisit the previous notebook which walks you through how to generate that plugin.

In [ ]:
%%bash

# Verification
geoips ls workflows -p cool_plugins

Please confirm that `My-ABI-Severe-Storms.yaml` shows up in your workflows

Introduce the "output side" of plugin types and their roles: output_formatter, filename_formatter, title_formatter, gridline_annotator, feature_annotator. Call out which already appear in their workflow and which are missing (no filename/title formatter yet; annotators use default).

In [ ]:
# Refresh registries defensively
!geoips config create-registries
# .ipynb_checkpoints cleanup if it's still necessary

## Section 1 - Tour of Existing Output Formatters

Conceptual contrast of the three built-ins by what they emit: netcdf_geoips (data), imagery_clean (bare PNG), imagery_annotated (PNG + title/gridlines/features). Emphasize: you usually pick an existing output formatter, you likely don't need to write one.

Establish the baseline — run the workflow exactly as they left it

In [ ]:
%%bash

export CARTOPY_DATA_DIR=$HOME/cartopy

!geoips run My-ABI-Severe-Storms $GEOIPS_TESTDATA_DIR/test_data_abi/data/goes16_20200918_1950/*

In [ ]:
# display the baseline PNG


Point at the output_formatter step in My-ABI-Severe-Storms.yaml. Instruction to change its name: from imagery_annotated to netcdf_geoips for a quick "data output" demo. Note the tradeoff (RGB→NetCDF is a demo, not the focus) and that they'll change it back.

In [ ]:
%%bash

export CARTOPY_DATA_DIR=$HOME/cartopy

# Re-run the workflow (now emitting NetCDF).
!geoips run My-ABI-Severe-Storms $GEOIPS_TESTDATA_DIR/test_data_abi/data/goes16_20200918_1950/*

In [ ]:
# Inspect the resulting .nc (open with xarray and print, or ncdump -h).


Instruction to revert the output_formatter step back to imagery_annotated.

EXPERIMENT - Invite attendees to swap the output formatter among the three built-ins, re-run, and compare.

In [ ]:
# Empty scratch cell

## Section 2 - Compose a Filename Formatter

Motivation — the workflow currently hardcodes output_fnames. A filename formatter generates the path from data attributes instead.


Anatomy of a filename formatter: class-based plugin, subclasses BaseFilenameFormatterPlugin, the three top-level attrs (interface/family/name), family: xarray_area_product_to_filename, the call() contract (inputs available, returns a single filepath), ends with PLUGIN_CLASS. Reference basic_fname as the model to copy from.


In [ ]:
# Create the target dir and empty file
mkdir -p .../plugins/classes/filename_formatters
touch .../plugins/classes/filename_formatters/my_new_filename_formatter.py

Clickable link to open the new file. 

Descriptive instructions for the class skeleton they should write (docstring, imports, class + the three attrs, PLUGIN_CLASS) — describe what each piece is.


Descriptive instructions for the call() body — build a path/filename from start_datetime, platform_name, source_name, product_name, area_def.area_id, chosen base dir; return the joined path.

In [ ]:
!geoips config create-registries
!geoips describe filename_formatter <name>

EXPERIMENT - invite attendees to invent their own naming scheme (subdirectories, extra fields, timestamp format, output extension).


In [ ]:
# Empty scratch cell

## Section 3 - Compose a Title Formatter

Motivation — control the text burned onto the annotated image instead of the default title. Note titles only surface on annotated imagery (tie back to Section 1).


Anatomy of a title formatter: subclasses BaseTitleFormatterPlugin, family: standard, the call() signature (area_def, xarray_obj, product_name_title, optional bg/copyright args), returns a title string. Reference static_standard as the model.


In [ ]:
%%bash

mkdir -p .../plugins/classes/title_formatters
touch .../plugins/classes/filename_formatters/my_new_title_formatter.py

Clickable link + descriptive instructions for the class skeleton (attrs, PLUGIN_CLASS).

Descriptive instructions for the call() body — assemble title lines from product_name_title, the xarray start_datetime, and a copyright string; return the composed multi-line string. Describe, don't supply.


In [ ]:
!geoips config create-registries
!geoips describe filename_formatter <name>

EXPERIMENT — multi-line titles, custom branding/copyright, reformatted timestamps.


In [ ]:
# Empty scratch cell

## Section 4 - Bringing it all together

Framing — assemble the two new plugins into My-ABI-Severe-Storms.yaml so that we can see the resulting image.


Descriptive edit instructions for the workflow YAML:
  * Add a filename_formatter step (kind: filename_formatter, name: your plugin).
  * Add a title_formatter step (kind: title_formatter, name: your plugin). 
  * Add both to the output_formatter step's depends_on.
  * Remove the hardcoded output_fnames entry so the filename formatter drives the path.


In [ ]:
# Rebuild registries
!geoips config create-registries

In [ ]:
!geoips run My-ABI-Severe-Storms ...

In [ ]:
# Display the resulting PNG and print/grep the generated output path (from the SINGLESOURCESUCCESS log line) to prove the filename formatter drove it.


Verification checklist:filename matches your scheme, custom title appears on the image.


PLAY BLOCK — combine variations from Sections 2 & 3, re-run.

In [ ]:
# Empty scratch cell


## Section 5 - Stretch: Gridline & Feature Annotators

Framing — these are YAML plugins, a different composition idiom from the Python plugins above.
The workflow already has retrieve_gridlines/retrieve_features steps using name: default.


In [ ]:
%%bash

mkdir -p $MY_PKG_DIR/plugins/yaml/gridline_annotators
cp ./updated_files/feature_annotators/tutorial.yaml $MY_PKG_DIR/plugins/yaml/gridline_annotators

mkdir -p $MY_PKG_DIR/plugins/yaml/feature_annotators
cp ./updated_files/feature_annotators/tutorial.yaml $MY_PKG_DIR/plugins/yaml/feature_annotators

Descriptive edit instructions — gridline YAML (spacing, line color/width/style, label options) and feature YAML (coastline/border/state styling). Describe the knobs, not the values.

Instruction to point the workflow's retrieve_gridlines/retrieve_features step name: fields at your new YAML plugins.


In [ ]:
%%bash
# Rebuild registries
geoips config create-registries

In [ ]:
!geoips run My-ABI-Severe-Storms ...

In [ ]:
# Display the new PNG and compare against the Section 4 output.

EXPERIMENT - Restyle the Gridline & Feature annotators freely

In [ ]:
# Empty scratch cell

## Section 6 - Wrap-up

Recap of everything built (filename formatter, title formatter, wired workflow, optional annotators). Pointer to the solutions branch (analogous to the beginner notebook's workshop-2026-solutions) and links for further reading.
